In [28]:
# Import Libraries
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

# Headers
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9"
}

# ---------------- LISTS ----------------

movie_name = []
year = []
rating = []
runtime = []
description = []
genres = []
movie_links = []

# To remove duplicates
visited = set()

# ---------------- SCRAPE MULTIPLE PAGES ----------------

for page in range(1, 6):   # 5 pages ≈ 100 movies

    url = f"https://www.themoviedb.org/movie/top-rated?page={page}"

    response = requests.get(url, headers=headers)

    soup = BeautifulSoup(response.content, "html.parser")

    movies = soup.find_all("a", href=True)

    for movie in movies:

        href = movie.get("href")

        # Match only movie URLs
        if href and re.match(r"^/movie/\d+", href):

            title = movie.text.strip()

            # Skip empty titles
            if title == "":
                continue

            # Full movie link
            full_link = "https://www.themoviedb.org" + href

            # Remove duplicates
            if full_link in visited:
                continue

            visited.add(full_link)

            print("Scraping:", title)

            # ---------------- OPEN MOVIE PAGE ----------------

            movie_response = requests.get(full_link, headers=headers)

            movie_soup = BeautifulSoup(movie_response.content, "html.parser")

            # ---------------- MOVIE NAME ----------------

            movie_name.append(title)

            # ---------------- YEAR ----------------

            release = movie_soup.find("span", class_="release")

            if release:

                release_text = release.text.strip()

                match = re.search(r"\d{4}", release_text)

                if match:
                    year.append(match.group())
                else:
                    year.append("N/A")

            else:
                year.append("N/A")

            # ---------------- RATING ----------------

            rate = movie_soup.find("div", class_="user_score_chart")

            if rate:
                rating.append(rate.get("data-percent"))
            else:
                rating.append("N/A")

            # ---------------- RUNTIME ----------------

            run = movie_soup.find("span", class_="runtime")

            if run:
                runtime.append(run.text.strip())
            else:
                runtime.append("N/A")

            # ---------------- DESCRIPTION ----------------

            desc = movie_soup.select_one("div.overview p")

            if desc:
                description.append(desc.text.strip())
            else:
                description.append("N/A")

            # ---------------- GENRES ----------------

            genre_tags = movie_soup.select("span.genres a")

            genre_list = []

            for g in genre_tags:
                genre_list.append(g.text.strip())

            genres.append(", ".join(genre_list))

            # ---------------- MOVIE LINK ----------------

            movie_links.append(full_link)

            # Delay to avoid blocking
            time.sleep(1)




Scraping: Swapped
Scraping: The Shawshank Redemption
Scraping: The Godfather
Scraping: Project Hail Mary
Scraping: The Godfather Part II
Scraping: Schindler's List
Scraping: 12 Angry Men
Scraping: Spirited Away
Scraping: The Dark Knight
Scraping: Remarkably Bright Creatures
Scraping: Dilwale Dulhania Le Jayenge
Scraping: The Green Mile
Scraping: The Punisher: One Last Kill
Scraping: The Lord of the Rings: The Return of the King
Scraping: Selena Gomez: My Mind & Me
Scraping: Parasite
Scraping: ¿Quieres ser mi hijo?
Scraping: Pulp Fiction
Scraping: Your Name.
Scraping: Interstellar
Scraping: The Good, the Bad and the Ugly
Scraping: Forrest Gump
Scraping: GoodFellas
Scraping: Seven Samurai
Scraping: Grave of the Fireflies
Scraping: Life Is Beautiful
Scraping: Fight Club
Scraping: The Lord of the Rings: The Fellowship of the Ring
Scraping: David Attenborough: A Life on Our Planet
Scraping: Harakiri
Scraping: City of God
Scraping: Human
Scraping: Cinema Paradiso
Scraping: The Lord of the Ri

In [53]:
# ---------------- CREATE DATAFRAME ----------------

movie_df = pd.DataFrame({
    "Movie Name": movie_name,
    "Year": year,
    "Rating": rating,
    "Runtime": runtime,
    "Genres": genres,
    "Description": description,
    "Movie Link": movie_links
})



In [54]:

# ---------------- SHOW DATA ----------------

print(movie_df.head())

print("Total Movies Scraped:", len(movie_df))

                 Movie Name  Year Rating Runtime  \
0                   Swapped  2026     90  1h 42m   
1  The Shawshank Redemption  1994     87  2h 22m   
2             The Godfather  2025     87  2h 55m   
3         Project Hail Mary  2026     86  2h 37m   
4     The Godfather Part II  1974     86  3h 22m   

                                  Genres  \
0  Animation, Family, Adventure, Fantasy   
1                           Drama, Crime   
2                           Drama, Crime   
3             Science Fiction, Adventure   
4                           Drama, Crime   

                                         Description  \
0  A small woodland creature and a majestic bird,...   
1  Imprisoned in the 1940s for the double murder ...   
2  Spanning the years 1945 to 1955, a chronicle o...   
3  Science teacher Ryland Grace wakes up on a spa...   
4  In the continuing saga of the Corleone crime f...   

                                          Movie Link  
0   https://www.themoviedb.org

In [55]:
# ---------------- SAVE CSV ----------------

movie_df.to_csv("TMDB_Movies.csv", index=False)

print("CSV File Saved Successfully")


CSV File Saved Successfully


In [56]:
# ---------------- SAVE excel ----------------

movie_df.to_excel("TMDB_Movies.xlsx", index=False)

print("Excel File Saved Successfully")

Excel File Saved Successfully


In [57]:
movie_df

,Movie Name,Year,Rating,Runtime,Genres,Description,Movie Link
0,Swapped,2026,90,1h 42m,"Animation, Family, Adventure, Fantasy","A small woodland creature and a majestic bird,...",https://www.themoviedb.org/movie/1007757-swapped
1,The Shawshank Redemption,1994,87,2h 22m,"Drama, Crime",Imprisoned in the 1940s for the double murder ...,https://www.themoviedb.org/movie/278-the-shaws...
2,The Godfather,2025,87,2h 55m,"Drama, Crime","Spanning the years 1945 to 1955, a chronicle o...",https://www.themoviedb.org/movie/238-the-godfa...
3,Project Hail Mary,2026,86,2h 37m,"Science Fiction, Adventure",Science teacher Ryland Grace wakes up on a spa...,https://www.themoviedb.org/movie/687163-projec...
4,The Godfather Part II,1974,86,3h 22m,"Drama, Crime",In the continuing saga of the Corleone crime f...,https://www.themoviedb.org/movie/240-the-godfa...
...,...,...,...,...,...,...,...
95,A Brighter Summer Day,1991,83,3h 57m,"Crime, Drama, Romance","A boy experiences first love, friendships and ...",https://www.themoviedb.org/movie/15804
96,City Lights,1931,82,1h 27m,"Comedy, Drama, Romance",A tramp falls in love with a beautiful blind f...,https://www.themoviedb.org/movie/901-city-lights
97,The Matrix,1999,82,2h 16m,"Action, Science Fiction","Set in the 22nd century, The Matrix tells the ...",https://www.themoviedb.org/movie/603-the-matrix
98,Michael Jackson's Thriller,1983,82,14m,"Horror, Thriller, Music",A night at the movies turns terrifying when Mi...,https://www.themoviedb.org/movie/92060-michael...


In [58]:
movie_df.info(5)


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Movie Name   100 non-null    str  
 1   Year         100 non-null    str  
 2   Rating       100 non-null    str  
 3   Runtime      100 non-null    str  
 4   Genres       100 non-null    str  
 5   Description  100 non-null    str  
 6   Movie Link   100 non-null    str  
dtypes: str(7)
memory usage: 5.6 KB


In [59]:
#---------------to convert str to int64---------------------------
cols = ["Year", "Rating"]

movie_df[cols] = movie_df[cols].apply(pd.to_numeric)

In [60]:
movie_df.info(5)

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Movie Name   100 non-null    str  
 1   Year         100 non-null    int64
 2   Rating       100 non-null    int64
 3   Runtime      100 non-null    str  
 4   Genres       100 non-null    str  
 5   Description  100 non-null    str  
 6   Movie Link   100 non-null    str  
dtypes: int64(2), str(5)
memory usage: 5.6 KB
